# 🔌 Day 03a: Dependency Inversion Principle — Depend on Abstractions

---

## 🎯 What You'll Master Today
- DIP: High-level modules shouldn't depend on low-level modules
- Dependency Injection — the practical implementation
- Constructor, method, and property injection
- Build loosely coupled, testable systems

---

## 🎭 The Analogy

Think of **charging your phone**:

- ❌ **Without DIP:** Your phone has a wire soldered directly to one specific wall outlet. Want to charge elsewhere? Rewire!
- ✅ **With DIP:** Your phone has a USB port (abstraction). Any charger that fits the port works — wall, car, power bank.

The phone (high-level) depends on the **USB interface** (abstraction), not on any specific charger (low-level).

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  DEPENDENCY INVERSION PRINCIPLE                                  ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Rule 1: High-level modules should NOT depend on low-level      ║
║          modules. Both should depend on ABSTRACTIONS.            ║
║                                                                   ║
║  Rule 2: Abstractions should NOT depend on details.             ║
║          Details should depend on abstractions.                  ║
║                                                                   ║
║  ❌ Without DIP:           ✅ With DIP:                         ║
║  OrderService              OrderService                         ║
║       │                         │                                ║
║       ▼                         ▼                                ║
║  MySQLDatabase            DatabaseInterface (ABC)               ║
║  (hard-coded)              /          \                          ║
║                        MySQL      PostgreSQL                    ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. DIP VIOLATION — Hard-coded Dependencies
# ============================================

# ❌ BAD — OrderService depends directly on MySQLDatabase

class MySQLDatabase:
    def save(self, data: dict):
        print(f"MySQL: Saving {data}")

class StripePayment:
    def charge(self, amount: float):
        print(f"Stripe: Charging ${amount}")


class OrderService:
    def __init__(self):
        # ❌ Creating dependencies internally — tightly coupled!
        self.db = MySQLDatabase()       # What if we switch to PostgreSQL?
        self.payment = StripePayment()  # What if we switch to Razorpay?
    
    def place_order(self, order: dict):
        self.payment.charge(order["total"])
        self.db.save(order)


service = OrderService()
service.place_order({"item": "Laptop", "total": 80000})

print("\n❌ Problems:")
print("   - Can't switch database without modifying OrderService")
print("   - Can't switch payment without modifying OrderService")
print("   - Can't unit test OrderService without hitting real DB/payment")

In [ ]:
# ============================================
# 2. DIP FIXED — Depend on Abstractions
# ============================================
from abc import ABC, abstractmethod

# Step 1: Define abstractions
class Database(ABC):
    @abstractmethod
    def save(self, data: dict) -> None:
        pass

class PaymentGateway(ABC):
    @abstractmethod
    def charge(self, amount: float) -> bool:
        pass

# Step 2: Implement concrete classes
class MySQLDatabase(Database):
    def save(self, data: dict):
        print(f"  💾 MySQL: Saved {data}")

class PostgreSQLDatabase(Database):
    def save(self, data: dict):
        print(f"  💾 PostgreSQL: Saved {data}")

class StripePayment(PaymentGateway):
    def charge(self, amount: float) -> bool:
        print(f"  💳 Stripe: Charged ₹{amount:,.0f}")
        return True

class RazorpayPayment(PaymentGateway):
    def charge(self, amount: float) -> bool:
        print(f"  💳 Razorpay: Charged ₹{amount:,.0f}")
        return True


# Step 3: High-level module depends on abstractions
class OrderService:
    def __init__(self, db: Database, payment: PaymentGateway):
        # ✅ Dependencies INJECTED — not created internally
        self.db = db
        self.payment = payment
    
    def place_order(self, order: dict):
        if self.payment.charge(order["total"]):
            self.db.save(order)
            print("  ✅ Order placed!")


# Easily swap implementations!
print("=== Using MySQL + Stripe ===")
service1 = OrderService(MySQLDatabase(), StripePayment())
service1.place_order({"item": "Laptop", "total": 80000})

print("\n=== Using PostgreSQL + Razorpay ===")
service2 = OrderService(PostgreSQLDatabase(), RazorpayPayment())
service2.place_order({"item": "Phone", "total": 50000})

print("\n✅ OrderService code didn't change at all!")

In [ ]:
# ============================================
# 3. Dependency Injection — Three Ways
# ============================================
from abc import ABC, abstractmethod

class Logger(ABC):
    @abstractmethod
    def log(self, message: str): pass

class ConsoleLogger(Logger):
    def log(self, message: str):
        print(f"  [CONSOLE] {message}")

class FileLogger(Logger):
    def log(self, message: str):
        print(f"  [FILE] {message}")


# WAY 1: Constructor Injection (most common, recommended)
class ServiceA:
    def __init__(self, logger: Logger):
        self._logger = logger
    
    def do_work(self):
        self._logger.log("ServiceA doing work (constructor injection)")


# WAY 2: Method Injection (when dependency varies per call)
class ServiceB:
    def do_work(self, logger: Logger):
        logger.log("ServiceB doing work (method injection)")


# WAY 3: Property Injection (when dependency is optional)
class ServiceC:
    def __init__(self):
        self._logger: Logger | None = None
    
    @property
    def logger(self) -> Logger | None:
        return self._logger
    
    @logger.setter
    def logger(self, logger: Logger):
        self._logger = logger
    
    def do_work(self):
        if self._logger:
            self._logger.log("ServiceC doing work (property injection)")


# Demo all three
print("1. Constructor Injection:")
ServiceA(ConsoleLogger()).do_work()

print("\n2. Method Injection:")
ServiceB().do_work(FileLogger())

print("\n3. Property Injection:")
sc = ServiceC()
sc.logger = ConsoleLogger()
sc.do_work()

In [ ]:
# ============================================
# 4. DIP Makes Testing Easy — Mock Dependencies
# ============================================
from abc import ABC, abstractmethod

class EmailService(ABC):
    @abstractmethod
    def send(self, to: str, subject: str, body: str) -> bool:
        pass

# Real implementation
class SMTPEmailService(EmailService):
    def send(self, to: str, subject: str, body: str) -> bool:
        print(f"  📧 SMTP: Sending '{subject}' to {to}")
        return True

# Test mock — no real emails sent!
class MockEmailService(EmailService):
    def __init__(self):
        self.sent_emails = []  # track what was "sent"
    
    def send(self, to: str, subject: str, body: str) -> bool:
        self.sent_emails.append({"to": to, "subject": subject})
        return True


class UserRegistration:
    def __init__(self, email_service: EmailService):
        self.email_service = email_service
    
    def register(self, name: str, email: str):
        # ... save user to DB ...
        self.email_service.send(email, "Welcome!", f"Hi {name}!")
        return True


# In production:
print("Production:")
reg_prod = UserRegistration(SMTPEmailService())
reg_prod.register("Sarthak", "sarthak@mail.com")

# In tests — no real emails!
print("\nTesting:")
mock_email = MockEmailService()
reg_test = UserRegistration(mock_email)
reg_test.register("TestUser", "test@mail.com")

# Verify behavior without sending real emails
assert len(mock_email.sent_emails) == 1
assert mock_email.sent_emails[0]["to"] == "test@mail.com"
print(f"  ✅ Mock captured {len(mock_email.sent_emails)} email(s)")
print(f"  ✅ Sent to: {mock_email.sent_emails[0]['to']}")

## 🗺️ DIP Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  DEPENDENCY INVERSION — QUICK REFERENCE                         ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  DIP: Both high-level and low-level depend on ABSTRACTIONS      ║
║                                                                   ║
║  Injection types:                                                ║
║    Constructor (recommended) → __init__(self, dep: ABC)         ║
║    Method → do_work(self, dep: ABC)                             ║
║    Property → self.dep = concrete_instance                      ║
║                                                                   ║
║  Benefits:                                                       ║
║    • Swap implementations without changing high-level code       ║
║    • Easy testing with mocks                                     ║
║    • Loose coupling between components                           ║
║                                                                   ║
║  DIP ≠ DI: DIP is the principle, DI is the technique            ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions — Practice These

| # | Question | Key Point |
|---|----------|------------|
| 1 | What is DIP? | High-level and low-level modules both depend on abstractions. Details depend on abstractions |
| 2 | DIP vs DI? | DIP = design principle (what). DI = implementation technique (how) — inject dependencies |
| 3 | Why is DI useful for testing? | Inject mock implementations — test without real DB, API calls, or email |
| 4 | Three types of injection? | Constructor (recommended), Method (per-call), Property (optional) |
| 5 | What problem does DIP solve? | Tight coupling. Without DIP, changing DB/email/payment means modifying business logic |
| 6 | How does FastAPI use DI? | Depends() function injects dependencies into route handlers — DB sessions, auth, etc. |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • DIP: depend on abstractions, not concretions          │
## │  • DI = inject dependencies, don't create them inside    │
## │  • Constructor injection = most common and recommended   │
## │  • Enables easy testing with mocks                       │
## │  • Enables swapping implementations without code changes │
## │  • SOLID complete: S-R-P, O-C-P, L-S-P, I-S-P, D-I-P  │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Creational Patterns — Singleton & Factory** — controlling object creation